<a href="https://colab.research.google.com/github/michalis0/DataScience_and_MachineLearning/blob/master/Assignements/Part%206/Assignment_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### DSML investigation

You are part of the Suisse Impossible Mission Force, or SIMF for short. You need to uncover a rogue agent that is trying to steal sensitive information.

Your mission, should you choose to accept it, is to find that agent before stealing any classified information. Good luck!

# Assignment part Six

### Mission Briefing

Welcome back to the **SIMF (Swiss Impossible Mission Force)**. Your objective remains the same: find the rogue agent before sensitive information is compromised.

### New Intelligence
We have received new intelligence that the rogue agent has attended various pop concerts. SIMF has tried checking the history of our remaining suspects on spotify, however none of them have any pop songs.

We need to dig deeper. To do so, we will build a basic recommender system using the spotify data to identify which of the remaining suspects is most likely to enjoy pop music.

#### Suspects remaining:
- **628854**
- **410319**
- **785994**






### Available Data
The only data you have is the spotify listening history of the suspects, and a list of user interactions with various songs.
The data may already contain some of the suspect interactions.

### Build a Recommender System
Using the data you have been provided, build a basic recommender system to identify which of the remaining suspects is most likely to enjoy pop music.

### Hints:
- Filter out the interactions data to not include the "skip" interactions and the interactions with a rating less than 3.
- Use the filtered interactions data to build a user-item matrix.
- Use cosine similarity to find songs similar to pop songs.
- Remove features that are not necessary for the recommender system.

## 1. Getting to know our data
- Interaction_ID: Unique identifier for each interaction
- User_ID: Unique identifier for each user
- File_Type: Type of file (e.g., audio, video)
- Interaction_Type: (Listen, Skip, Share, Like)
- Rating: User rating for the interaction (1-5 scale)
- Timestamp: Date and time of the interaction
- Age: Age of the user
- Gender: Gender of the user
- Subscription_Type: Type of subscription (Free, Premium)
- Title: Title of the song
- Artist: Artist of the song
- Album: Album of the song
- Genre: Genre of the song
- Release_Year: Year the song was released
- Lyrics: Lyrics of the song
- File_id: Unique identifier for each file


In [1]:
import pandas as pd
import os
# download the dataset from github
if os.path.exists("train_data.csv") == False:
    !wget https://raw.githubusercontent.com/michalis0/DataScience_and_MachineLearning/refs/heads/master/Assignements/Part%206/train_data.csv
data = pd.read_csv("train_data.csv")
data.tail(10)



--2025-11-11 20:49:26--  https://raw.githubusercontent.com/michalis0/DataScience_and_MachineLearning/refs/heads/master/Assignements/Part%206/train_data.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 892269 (871K) [text/plain]
Saving to: ‘train_data.csv’

train_data.csv      100%[===================>] 871.36K  --.-KB/s    in 0.05s   

2025-11-11 20:49:26 (18.6 MB/s) - ‘train_data.csv’ saved [892269/892269]



,Interaction_ID,User_ID,File_Type,Interaction_Type,Rating,Timestamp,Age,Gender,Subscription_Type,Title,Artist,Album,Genre,Release_Year,Lyrics,File_ID,Sentiment,Target
4988,I04991,U0053,Audio,Listen,2,2025-08-31 01:58:30.422989,48,F,Premium,Audio_Song_1781,Dr. Dre,Album_61,Electronic,2012,I get by with a little help from my friends,A1781,0.416,0
4989,I04992,U0393,Audio,Share,4,2023-04-17 23:49:30.422989,54,Other,Free,Audio_Song_1809,Lorde,Album_94,Pop,1994,"Nothing else matters, never cared for what the...",A1809,0.548,1
4990,I04993,U0335,Video,Like,5,2023-02-19 11:55:30.422989,53,F,Premium,Video_Song_941,Tina Turner,Album_5,Classical,2003,"Rolling in the deep, you had my heart inside o...",V941,0.103,1
4991,I04994,U0342,Audio,Like,1,2022-12-29 13:33:30.422989,58,Other,Free,Audio_Song_2088,Tina Turner,Album_56,Classical,1999,"Cause baby you're a firework, come on show 'em...",A2088,-0.235,0
4992,I04995,U0161,Video,Skip,3,2023-03-23 16:15:30.422989,41,F,Premium,Video_Song_1282,U2,Album_92,Jazz,2012,"All the single ladies, all the single ladies",V1282,0.340,0
4993,I04996,U0289,Audio,Like,4,2024-01-31 15:23:30.422989,45,M,Premium,Audio_Song_587,Katy Perry,Album_87,Rock,2022,"I want to hold your hand, I want to hold your ...",A587,-0.339,1
4994,I04997,U0311,Video,Listen,1,2024-10-22 21:13:30.422989,15,M,Premium,Video_Song_2346,Queen,Album_46,Pop,1991,"Hello from the other side, I must have called ...",V2346,0.634,0
4995,I04998,U0411,Audio,Skip,3,2024-01-17 03:22:30.422989,53,M,Free,Audio_Song_2357,Travis Scott,Album_19,Rock,1995,"Don't stop believin', hold on to that feeling",A2357,-0.015,0
4996,I04999,U0110,Audio,Skip,1,2024-03-13 01:10:30.422989,19,M,Premium,Audio_Song_1745,Miley Cyrus,Album_83,Hip-Hop,2025,"Cause baby you're a firework, come on show 'em...",A1745,-0.464,0
4997,I05000,U0288,Audio,Listen,4,2023-06-10 14:30:30.422989,48,F,Premium,Audio_Song_1089,Lady Gaga,Album_10,Classical,2018,"Nothing else matters, never cared for what the...",A1089,-0.109,1


### Data cleaning keep only the interactions with a rating 3 or above and that do not consist in a 'skip'

In [3]:
### your code here ###
filtered_data = data[
    (data['Interaction_Type'].str.lower() != 'skip') &
    (data['Rating'] >= 3)
]

print("Original data:", len(data))
print( "Filtered data:", len(filtered_data))
filtered_data.head()



Original data: 4998
Filtered data: 2218


,Interaction_ID,User_ID,File_Type,Interaction_Type,Rating,Timestamp,Age,Gender,Subscription_Type,Title,Artist,Album,Genre,Release_Year,Lyrics,File_ID,Sentiment,Target
0,I00001,U0494,Audio,Listen,4,2023-06-05 00:05:30.389872,51,M,Premium,Audio_Song_1825,Bruno Mars,Album_90,Pop,2020,I get by with a little help from my friends,A1825,-0.735,1
2,I00003,U0224,Video,Share,4,2024-10-19 01:52:30.389872,58,M,Free,Video_Song_572,The Chainsmokers,Album_18,Electronic,2004,"Let it be, let it be, let it be, let it be",V572,0.420,1
6,I00007,U0427,Audio,Share,4,2025-02-04 18:39:30.389872,37,M,Premium,Audio_Song_2496,Coldplay,Album_55,Hip-Hop,2000,Hit me baby one more time,A2496,0.319,1
7,I00008,U0123,Audio,Like,5,2023-02-09 01:29:30.389872,21,Other,Premium,Audio_Song_530,Sam Cooke,Album_81,Classical,2024,"All the single ladies, all the single ladies",A530,0.479,1
13,I00014,785994,Video,Listen,3,2025-09-19 08:06:30.389872,41,M,Premium,Video_Song_660,Louis Armstrong,Album_45,Electronic,1996,"Beat it, beat it, no one wants to be defeated",V660,0.020,0


## 2. Build a user-item matrix
Now that we have cleaned the data, we can build a user-item matrix. This matrix will help us understand the interactions between users and songs.

In [11]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# build a user-item interaction matrix, you can use pivot_table from pandas



interaction_matrix =  filtered_data.pivot_table(
    index='User_ID',       # each row = a user
    columns='File_ID',     # each column = a song
    values='Rating',       # the values = user’s rating
    fill_value=0           # missing ratings replaced with 0
)


In [17]:
interaction_matrix.head()
#interaction_matrix.sum(axis=1).head()


File_ID,A002,A005,A009,A010,A013,A015,A016,A018,A021,A022,...,V974,V975,V977,V980,V983,V987,V990,V992,V994,V999
User_ID,,,,,,,,,,,,,,,,,,,,,
410319,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
628854,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
785994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
U0002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


# 4. identifying the recommendations for our 3 suspects
Now that we have built the recommender system, we can use it to identify which of the remaining suspects is most likely to enjoy pop music.
- For each suspect, use the songs they have interacted with.
- Find similar songs using the recommender system.
- We will use cosine similarity to find similar songs, and set k = 5.

- **628854** has listened to songs: Video_Song_1683, Audio_Song_2372, Audio_Song_280, Audio_Song_1825, Video_Song_1825, Video_Song_1006
- **410319** has listened to songs: Audio_Song_717, Audio_Song_2372, Audio_Song_1481, Video_Song_572
- **785994** has listened to songs: Audio_Song_1733, Audio_Song_389, Video_Song_2500, Video_Song_660



In [19]:
from sklearn.metrics.pairwise import cosine_similarity
# Compute item-item similarity matrix
item_similarity = cosine_similarity(interaction_matrix.T)

# Function to get top N recommendations for a given user
def get_recommendations(user_id, data_matrix, item_similarity, top_n=5):
    # Get the index of the user in the matrix
    if user_id not in data_matrix.index:
        print(f"User {user_id} not found.")
        return []    #pass
    # User’s rating vector (1 × n_items)
    user_ratings = data_matrix.loc[user_id].values.reshape(1, -1)

    # Predict ratings for all items
    # Multiply similarity matrix (items × items) by user_ratings (1 × items)
    predicted_scores = np.dot(user_ratings, item_similarity)

    # Convert back to a Series for easier handling
    predicted_scores = pd.Series(predicted_scores.flatten(), index=data_matrix.columns)

    # Remove songs the user has already rated (non-zero ratings)
    already_rated = data_matrix.loc[user_id]
    predicted_scores = predicted_scores[already_rated == 0]

    # Return the top N recommended items
    top_recommendations = predicted_scores.sort_values(ascending=False).head(top_n)

    return top_recommendations




### Find the recommendations for each suspect and identify which suspect is most likely to enjoy pop music.

In [24]:
### your code here ###
suspects = ['628854', '410319', '785994']

suspect_recommendations = {}
for user_id in suspects:
    recs = get_recommendations(user_id, interaction_matrix, item_similarity, top_n=5)
    if isinstance(recs, pd.Series):
        suspect_recommendations[user_id] = list(recs.index)
    else:
        suspect_recommendations[user_id] = []

print(suspect_recommendations)


{'628854': ['V999', 'A002', 'A005', 'A009', 'A010'], '410319': ['V999', 'A002', 'A005', 'A009', 'A010'], '785994': ['A1646', 'A1348', 'V2113', 'V1170', 'V1723']}


## Your investigation is complete. The SIMF task force extends their deepest gratitude for your unwavering dedication and service.

**Remember to complete the quiz and submit your code (this notebook) on Moodle before the deadline. Your mission isn’t over until all tasks are finished!**